# 05 — Independent URL Dataset Construction and Bias Audit

**Research question:** After finding severe structural shortcuts in PhiUSIIL and LegitPhish, can a separately collected URL corpus support more credible URL-only phishing modeling?

I use the **Wangchuk (2026) phishing URL dataset** (Common Crawl CC-MAIN-2024-33 benign URLs; PhishTank phishing URLs) as my **new primary corpus**. I audit, deduplicate, investigate overlap with LegitPhish, and create **hostname-disjoint** 70/15/15 modeling partitions. I do **not** merge LegitPhish into the new training corpus because its collection artifacts could reintroduce the bias I am trying to avoid. I do not touch the test set during model selection.

**Label convention:** 0 = legitimate; 1 = phishing. These historical labels are not real-time verdicts on the current safety of URLs.

## 1. Provenance and setup

I verify the externally published source's claimed dimensions and labels against the actual CSV. The source description reports 149,726 URLs: 94,919 legitimate and 54,807 phishing. The two classes originate from different collection mechanisms, which remains a potential confounder. No websites are fetched or visited by this notebook.

**Source:** Tandin Wangchuk (2026), *Phishing URL dataset*, Mendeley Data, V1, DOI: [10.17632/3jddhy2f6s.1](https://doi.org/10.17632/3jddhy2f6s.1), CC BY 4.0. Original LegitPhish dataset: DOI [10.17632/hx4m73v2sf.2](https://doi.org/10.17632/hx4m73v2sf.2).

In [1]:
from pathlib import Path
from urllib.parse import urlsplit
import hashlib
import json
import numpy as np
import pandas as pd

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROCESSED_DIR / "splits"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
for directory in (PROCESSED_DIR, SPLIT_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
INDEPENDENT_PATH = RAW_DIR / "dataset2.csv"
LEGITPHISH_PATH = PROCESSED_DIR / "legitphish_clean_labeled.csv"
assert INDEPENDENT_PATH.is_file(), f"Place dataset2.csv in {RAW_DIR}"
print("New source:", INDEPENDENT_PATH.name)
print("Project root:", PROJECT_ROOT)

New source: dataset2.csv
Project root: /mnt/data/url_dataset_final_project


## 2. Inspect and clean the independent source

I confirm the original labels against the source's documented class counts. I discard missing/invalid URL records if any; I never guess labels. I remove duplicate **exact URL strings** only after checking label consistency. I retain the source provenance with each record.

In [2]:
raw = pd.read_csv(INDEPENDENT_PATH, low_memory=False)
assert {"URL", "Label"}.issubset(raw.columns)
print("Raw shape:", raw.shape)
print("Missing values:", raw[["URL", "Label"]].isna().sum().to_dict())
print("Original labels:", raw["Label"].value_counts(dropna=False).to_dict())
assert raw.shape[0] == 149726, "Unexpected row count; review source version"
assert raw["Label"].value_counts().to_dict() == {0: 94919, 1: 54807}, "Verify label convention before continuing"
valid = raw.loc[raw["URL"].notna() & raw["Label"].isin([0, 1])].copy()
valid["URL"] = valid["URL"].astype(str).str.strip()
valid = valid.loc[valid["URL"].str.len().gt(0)]
conflicts = valid.groupby("URL")["Label"].nunique()
assert (conflicts <= 1).all(), "Conflicting URL labels: investigate before deduplication"
repeated = int(valid["URL"].duplicated().sum())
clean = (valid.drop_duplicates("URL", keep="first")
         .rename(columns={"URL": "url", "Label": "label"})[["url", "label"]]
         .reset_index(drop=True))
clean["label"] = clean["label"].astype("int8")
clean["source"] = "Wangchuk_2026_CommonCrawl_Phishtank"
# I exclude malformed URLs that cannot yield a hostname through the same parser used at inference.
def valid_hostname(url):
    try:
        candidate = url if "://" in url else "http://" + url
        return bool(urlsplit(candidate).hostname)
    except (ValueError, TypeError):
        return False
invalid_hostname_count = int((~clean.url.map(valid_hostname)).sum())
clean = clean.loc[clean.url.map(valid_hostname)].reset_index(drop=True)
print("Malformed hostname URLs excluded:", invalid_hostname_count)
print("Exact URL duplicates removed:", repeated)
print("Clean unique records:", len(clean))
print("Clean class counts:", clean["label"].value_counts().sort_index().to_dict())
assert clean["url"].is_unique and clean["label"].isin([0, 1]).all()

Raw shape: (149726, 2)
Missing values: {'URL': 0, 'Label': 0}
Original labels: {0: 94919, 1: 54807}


Malformed hostname URLs excluded: 2
Exact URL duplicates removed: 19950
Clean unique records: 129774
Clean class counts: {0: 74970, 1: 54804}


## 3. Structural bias audit

I calculate HTTPS, path, deep-path and query indicators from the actual URLs, **without** using any dataset-precomputed features. A root slash `/` counts as a path but **not** a deep path. I compare classes and evaluate a simple HTTP-or-path rule as a diagnostic, not as trained model performance.

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def structure(url: str) -> dict:
    candidate = url if "://" in url else "http://" + url
    try:
        parsed = urlsplit(candidate)
        hostname = (parsed.hostname or "").lower().rstrip(".")
        return {"hostname": hostname, "is_https": int(parsed.scheme.lower() == "https"),
                "has_path": int(bool(parsed.path)), "has_deep_path": int(len(parsed.path) > 1),
                "has_query": int(bool(parsed.query)), "url_length": len(url),
                "path_length": len(parsed.path)}
    except (ValueError, TypeError):
        return {"hostname": "", "is_https": 0, "has_path": 0, "has_deep_path": 0,
                "has_query": 0, "url_length": len(url), "path_length": 0}

struct = pd.DataFrame([structure(u) for u in clean["url"]])
assert struct["hostname"].ne("").all(), "Failed URL hostname parsing; review malformed examples"
audit = pd.concat([clean, struct], axis=1)
structure_summary = (audit.groupby("label")
    .agg(records=("label", "size"), https_rate=("is_https", "mean"),
         path_rate=("has_path", "mean"), deep_path_rate=("has_deep_path", "mean"),
         query_rate=("has_query", "mean"), average_url_length=("url_length", "mean"),
         average_path_length=("path_length", "mean"), unique_hosts=("hostname", "nunique")))
print("STRUCTURAL COMPARISON (0 = legitimate; 1 = phishing)")
display(structure_summary.round(4))
shortcut_pred = ((audit["is_https"] == 0) | (audit["has_path"] == 1)).astype(int)
shortcut_results = {"accuracy": accuracy_score(audit.label, shortcut_pred),
                    "precision": precision_score(audit.label, shortcut_pred, zero_division=0),
                    "recall": recall_score(audit.label, shortcut_pred, zero_division=0),
                    "f1": f1_score(audit.label, shortcut_pred, zero_division=0)}
print("HTTP-OR-PATH RULE (full-corpus diagnostic, not held-out):", shortcut_results)
print("Rule confusion matrix:", confusion_matrix(audit.label, shortcut_pred).tolist())
structure_summary.to_csv(OUTPUT_DIR / "url_v2_structure_by_class.csv")

STRUCTURAL COMPARISON (0 = legitimate; 1 = phishing)


,records,https_rate,path_rate,deep_path_rate,query_rate,average_url_length,average_path_length,unique_hosts
label,,,,,,,,
0,74970,0.8742,0.9884,0.9031,0.2753,87.0963,33.1701,4441
1,54804,0.8951,0.8982,0.5284,0.1172,66.5544,22.8277,32936


HTTP-OR-PATH RULE (full-corpus diagnostic, not held-out): {'accuracy': 0.39895510656988303, 'precision': 0.407763515770387, 'recall': 0.9355703963214363, 'f1': 0.5679771360207371}
Rule confusion matrix: [[501, 74469], [3531, 51273]]


## 4. Cross-source overlap and source acceptance

I measure overlap against the already cleaned LegitPhish source (if available). Overlap is useful for assessing independence but does **not** prove the new source is representative. I do not combine the two datasets: the prior LegitPhish corpus demonstrably contains severe class-conditional shortcuts. I use the new source as the primary dataset and keep all overlap findings in the report.

In [4]:
overlap = {"legitphish_available": bool(LEGITPHISH_PATH.is_file())}
if LEGITPHISH_PATH.is_file():
    prior = pd.read_csv(LEGITPHISH_PATH, usecols=["url", "label"])
    exact_urls = set(clean.url).intersection(set(prior.url))
    prior_hosts = set(prior.url.map(lambda u: structure(u)["hostname"]))
    hosts_shared = set(audit.hostname).intersection(prior_hosts)
    matched = clean[clean.url.isin(exact_urls)].merge(prior, on="url", suffixes=("_new", "_old"))
    overlap.update({"exact_url_overlap": len(exact_urls), "shared_hostnames": len(hosts_shared),
                    "label_conflicts_in_exact_overlap": int((matched.label_new != matched.label_old).sum())})
    print("LegitPhish cross-source overlap:", overlap)
else:
    print("LegitPhish clean extract not present; skip optional overlap check.")

LegitPhish cross-source overlap: {'legitphish_available': True, 'exact_url_overlap': 16, 'shared_hostnames': 133, 'label_conflicts_in_exact_overlap': 0}


## 5. Hostname-aware dataset construction

I split the **new source only**, grouping exact hostnames so they never appear in more than one partition. I use a reproducible group-level allocation that targets 70/15/15 **per original label** while keeping mixed-label hostname groups intact. Since some hostnames contain thousands of URLs, I assign larger groups first and balance cumulative class counts. This is a pre-model allocation strategy, not an optimization of test performance. It does not prevent related subdomains, shared infrastructures, or collection-source patterns from crossing partitions.

In [5]:
group_info = (audit.groupby("hostname")["label"]
              .agg(group_size="size", phishing_count="sum").reset_index())
group_info["legitimate_count"] = group_info.group_size - group_info.phishing_count
mixed_hosts = int(((group_info.phishing_count > 0) & (group_info.legitimate_count > 0)).sum())
print("Hostnames:", len(group_info), "Mixed-label hostnames:", mixed_hosts)
print("Largest hostname groups:")
display(group_info.sort_values("group_size", ascending=False).head(10))

proportions = np.array([0.70, 0.15, 0.15])
labels_total = np.array([int((audit.label == 0).sum()), int((audit.label == 1).sum())])
target = np.outer(proportions, labels_total.astype(float))
allocated = np.zeros((3, 2), dtype=float)
rng = np.random.default_rng(RANDOM_STATE)
info = group_info.copy()
info["tie_order"] = rng.random(len(info))
info = info.sort_values(["group_size", "tie_order"], ascending=[False, True])
partitions = ("train", "val", "test")
assigned_host = {}
for hostname, size, phishing, legitimate in info[["hostname", "group_size", "phishing_count", "legitimate_count"]].itertuples(index=False, name=None):
    majority = int(phishing >= legitimate)
    available_fraction = (target[:, majority] - allocated[:, majority]) / target[:, majority]
    overall_fraction = (target.sum(axis=1) - allocated.sum(axis=1)) / target.sum(axis=1)
    destination = int(np.argmax(available_fraction + 0.15 * overall_fraction + rng.random(3) * 1e-9))
    allocated[destination] += [legitimate, phishing]
    assigned_host[hostname] = partitions[destination]
audit["split"] = audit.hostname.map(assigned_host)
assert audit.split.notna().all()
split_frames = {name: audit.loc[audit.split.eq(name), ["url", "label", "source"]].reset_index(drop=True)
                for name in partitions}
summary = pd.DataFrame([{"split": name, "records": len(df),
                         "share_pct": 100 * len(df) / len(clean),
                         "phishing_pct": 100 * df.label.mean(),
                         "unique_hostnames": int((audit.split == name).sum())}
                        for name, df in split_frames.items()])
# I replace the count of URL rows with the correct distinct-hostname count.
summary["unique_hostnames"] = [audit.loc[audit.split.eq(name), "hostname"].nunique() for name in partitions]
print("HOSTNAME-DISJOINT SPLITS")
display(summary.round(3))

Hostnames: 37352 Mixed-label hostnames: 25
Largest hostname groups:


,hostname,group_size,phishing_count,legitimate_count
10772,cloudflare-ipfs.com,3220,3220,0
12669,docs.google.com,3165,3165,0
17211,ipfs.eth.aragon.network,2180,2180,0
10346,cf-ipfs.com,1354,1354,0
21366,new.express.adobe.com,949,949,0
30830,tinyurl.com,932,650,282
17213,ipfs.io,877,877,0
28682,sites.google.com,658,658,0
8645,bit.ly,457,169,288
14556,forms.gle,313,17,296


HOSTNAME-DISJOINT SPLITS


,split,records,share_pct,phishing_pct,unique_hostnames
0,train,90840,69.999,42.23,26617
1,val,19467,15.001,42.23,5355
2,test,19467,15.001,42.23,5380


## 6. Validation, saving and manifest

I verify exact row coverage, URL uniqueness, group disjointness, and label distributions before saving. I name these files `url_v2_*` (CSV, plus Parquet when available) so the earlier **biased PhiUSIIL splits remain untouched**. I retain source, code version/context, label mapping, known confounders, and checksum in the manifest. The held-out test partition is not used to select a model.

In [6]:
assert sum(len(df) for df in split_frames.values()) == len(clean)
assert pd.concat([df.url for df in split_frames.values()], ignore_index=True).is_unique
host_sets = {name: set(audit.loc[audit.split.eq(name), "hostname"]) for name in partitions}
assert host_sets["train"].isdisjoint(host_sets["val"])
assert host_sets["train"].isdisjoint(host_sets["test"])
assert host_sets["val"].isdisjoint(host_sets["test"])
for name, df in split_frames.items():
    assert df.label.isin([0, 1]).all()
    path = SPLIT_DIR / f"url_v2_{name}.csv"
    df.to_csv(path, index=False)
    pd.testing.assert_frame_equal(df, pd.read_csv(path), check_dtype=False)
    print(f"{name.upper()}: {len(df):,} CSV saved and verified")

clean.to_csv(PROCESSED_DIR / "url_v2_clean.csv", index=False)
# I also save Parquet copies when the optional engine is installed.
try:
    import pyarrow
except ImportError:
    print("Optional pyarrow is unavailable here; verified CSV files are ready.")
else:
    for name, df in split_frames.items():
        df.to_parquet(SPLIT_DIR / f"url_v2_{name}.parquet", index=False)
    clean.to_parquet(PROCESSED_DIR / "url_v2_clean.parquet", index=False)
    print("Parquet copies saved.")
summary.to_csv(OUTPUT_DIR / "url_v2_split_summary.csv", index=False)
manifest = {
  "dataset": "Wangchuk 2026, Mendeley V1, DOI 10.17632/3jddhy2f6s.1",
  "source_method": "Common Crawl CC-MAIN-2024-33 benign; PhishTank phishing",
  "source_file_sha256": hashlib.sha256(INDEPENDENT_PATH.read_bytes()).hexdigest(),
  "original_label_mapping": {"0": "legitimate", "1": "phishing"},
  "raw_rows": int(len(raw)), "exact_url_duplicates_removed": int(repeated), "malformed_hostname_urls_excluded": invalid_hostname_count,
  "clean_rows": int(len(clean)), "label_counts_clean": clean.label.value_counts().sort_index().to_dict(),
  "unique_hostnames": int(group_info.shape[0]), "mixed_label_hostnames": mixed_hosts,
  "split_seed": RANDOM_STATE, "split_policy": "hostname disjoint; size-aware greedy stratified by label",
  "split_sizes": {key: int(len(df)) for key, df in split_frames.items()},
  "source_overlap_with_legitphish": overlap,
  "full_dataset_shortcut_diagnostic": shortcut_results,
  "limitations": [
      "The two classes come from different data collection sources and may retain source-specific artifacts.",
      "Hostname separation does not ensure registrable-domain or URL-template separation.",
      "Historical labels and URL-only features do not verify current website maliciousness.",
      "Test partition must remain untouched until URL model selection is finalized."
  ]
}
with (OUTPUT_DIR / "url_v2_dataset_manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("DATASET CONSTRUCTION: COMPLETE — with documented limitations")

TRAIN: 90,840 CSV saved and verified
VAL: 19,467 CSV saved and verified
TEST: 19,467 CSV saved and verified


Optional pyarrow is unavailable here; verified CSV files are ready.
DATASET CONSTRUCTION: COMPLETE — with documented limitations


## 7. Conclusion and next steps

I confirmed that this independently collected source contains both HTTPS phishing URLs and legitimate URLs with non-empty paths, unlike the near-deterministic patterns found in PhiUSIIL and LegitPhish. The naive HTTP-or-path rule no longer separates the classes well, so this source offers a more useful starting point for URL-only modeling.

I still cannot claim that the source is free of dataset bias: benign URLs come from Common Crawl while phishing URLs come from PhishTank. Large repeated-hostname groups and possible public-hosting patterns also warrant careful interpretation. I keep the previous datasets as **audited references**, not blended training data. In `06_url_modeling.ipynb`, I will fit and compare models on `url_v2_train` and `url_v2_val`, examine structural subgroups, and evaluate `url_v2_test` **once** after selecting the final configuration. I will clearly distinguish historical benchmark performance from real-world phishing detection guarantees.

## Post-Split Diagnostics — Hostname Concentration and Distribution Shift

During final model evaluation, I observed a substantial difference between validation and test performance.

My Histogram Gradient Boosting classifier achieved 91.79% phishing F1-score on validation but only 67.50% on the held-out test partition.

Further investigation showed that query-containing phishing URLs accounted for approximately 6.79% of training phishing records, 5.77% of validation records, and 40.65% of test records.

This difference raises questions about the representativeness of my hostname-aware dataset partitions.

I investigate whether a small number of large hostname groups contributed disproportionately to the test partition.

This is a retrospective investigation. I preserve the original partitions and results rather than modifying the test data to improve model performance.

In [3]:
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
import numpy as np

# I locate the project directory from my notebook.
PROJECT_ROOT = Path.cwd().resolve().parent

SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits"

# I load the original URL partitions without modifying them.
partition_frames = {
    "Train": pd.read_csv(SPLIT_DIR / "url_v2_train.csv"),
    "Validation": pd.read_csv(SPLIT_DIR / "url_v2_val.csv"),
    "Test": pd.read_csv(SPLIT_DIR / "url_v2_test.csv")
}

for name, frame in partition_frames.items():
    print(f"{name}: {frame.shape}")

Train: (90840, 3)
Validation: (19467, 3)
Test: (19467, 3)


In [5]:
# I extract hostnames and identify query-containing phishing URLs.
def get_hostname(url):
    try:
        return (urlsplit(str(url)).hostname or "").lower()
    except ValueError:
        return ""


concentration_results = []
top_hostnames = {}

for partition_name, frame in partition_frames.items():

    # I work on a temporary copy to preserve the original data.
    data = frame.copy()

    data["hostname"] = data["url"].map(get_hostname)

    data["has_query"] = data["url"].map(
        lambda url: bool(urlsplit(str(url)).query)
    )

    # I isolate phishing URLs containing query parameters.
    subset = data.loc[
        (data["label"] == 1) &
        (data["has_query"])
    ]

    hostname_counts = (
        subset["hostname"]
        .value_counts()
    )

    total = len(subset)

    concentration_results.append({
        "partition": partition_name,
        "query_phishing_urls": total,
        "unique_hostnames": hostname_counts.size,
        "largest_hostname_count": (
            int(hostname_counts.iloc[0])
            if len(hostname_counts) else 0
        ),
        "top_5_share": (
            hostname_counts.head(5).sum() / total
            if total else 0
        ),
        "top_10_share": (
            hostname_counts.head(10).sum() / total
            if total else 0
        )
    })

    top_hostnames[partition_name] = (
        hostname_counts
        .head(10)
        .rename_axis("hostname")
        .reset_index(name="url_count")
    )


hostname_concentration = pd.DataFrame(
    concentration_results
)

print("QUERY-CONTAINING PHISHING URL CONCENTRATION")

display(hostname_concentration.round(4))

for partition_name, table in top_hostnames.items():

    print(f"\n{partition_name.upper()} — TOP 10 HOSTNAMES")

    display(table)

QUERY-CONTAINING PHISHING URL CONCENTRATION


,partition,query_phishing_urls,unique_hostnames,largest_hostname_count,top_5_share,top_10_share
0,Train,2605,1366,105,0.1317,0.1950
1,Validation,474,254,209,0.4578,0.4789
2,Test,3342,234,3098,0.9294,0.9324



TRAIN — TOP 10 HOSTNAMES


,hostname,url_count
0,tinyurl.com,105
1,rp.mockplus.com,71
2,forms.office.com,67
3,sites.google.com,56
4,43.134.167.94,44
5,43.159.37.67,35
6,129.226.210.78,35
7,43.156.237.181,34
8,43.156.7.24,31
9,43.153.207.103,30



VALIDATION — TOP 10 HOSTNAMES


,hostname,url_count
0,cloudflare-ipfs.com,209
1,bionicsolutionsllc.com,2
2,prodigal.ipower.com,2
3,realizesolucoefinanceiras.blogspot.com,2
4,securelogin.aepl.org.au,2
5,genoeneimoveis.com.br,2
6,unitednewsservice.com,2
7,white-ar.com,2
8,consulteaquirennejaneiro.lojaintegrada.com.br,2
9,www.getshuffler.com,2



TEST — TOP 10 HOSTNAMES


,hostname,url_count
0,docs.google.com,3098
1,luziaacesso.blogspot.com,2
2,acesso-luiza-rapido.blogspot.com,2
3,demo.dmv-driversed.com,2
4,sagestonelaw-1318439371.cos.eu-frankfurt.myqcl...,2
5,livenmitac-my.sharepoint.com,2
6,garena.freefiremax.hstn.me,2
7,onedrive.live.com,2
8,digitalluizaonline.blogspot.com,2
9,ersfilter-my.sharepoint.com,2


### Large Hostname Group Investigation

My hostname concentration analysis revealed that `docs.google.com` accounts for 3,098 query-containing phishing URLs in the test partition.

This single hostname represents approximately 92.7% of query-containing phishing URLs in the test set.

The result provides a plausible explanation for the unusual test distribution and the substantial reduction in phishing recall.

I investigate the group's label distribution and URL characteristics before deciding whether the partitioning strategy should be revised.

I preserve the original dataset annotations and avoid assuming that all Google Docs URLs are inherently malicious.

In [8]:
# I examine unusually large hostname groups across partitions.

selected_hosts = [
    "docs.google.com",
    "cloudflare-ipfs.com",
    "tinyurl.com"
]

hostname_results = []

for partition_name, frame in partition_frames.items():

    data = frame.copy()

    data["hostname"] = data["url"].map(get_hostname)

    data["has_query"] = data["url"].map(
        lambda url: bool(urlsplit(str(url)).query)
    )

    data["url_length"] = data["url"].str.len()

    for hostname in selected_hosts:

        group = data.loc[
            data["hostname"] == hostname
        ]

        hostname_results.append({
            "partition": partition_name,
            "hostname": hostname,
            "total_urls": len(group),
            "legitimate": int((group["label"] == 0).sum()),
            "phishing": int((group["label"] == 1).sum()),
            "query_rate": (
                group["has_query"].mean()
                if len(group) else np.nan
            ),
            "mean_url_length": (
                group["url_length"].mean()
                if len(group) else np.nan
            )
        })

selected_hostname_summary = pd.DataFrame(
    hostname_results
)

print("SELECTED HOSTNAME DISTRIBUTION")

display(selected_hostname_summary.round(4))

SELECTED HOSTNAME DISTRIBUTION


,partition,hostname,total_urls,legitimate,phishing,query_rate,mean_url_length
0,Train,docs.google.com,0,0,0,NaN,NaN
1,Train,cloudflare-ipfs.com,0,0,0,NaN,NaN
2,Train,tinyurl.com,932,282,650,0.1137,33.0343
3,Validation,docs.google.com,0,0,0,NaN,NaN
4,Validation,cloudflare-ipfs.com,3220,0,3220,0.0649,101.5922
5,Validation,tinyurl.com,0,0,0,NaN,NaN
6,Test,docs.google.com,3165,0,3165,0.9788,173.9681
7,Test,cloudflare-ipfs.com,0,0,0,NaN,NaN
8,Test,tinyurl.com,0,0,0,NaN,NaN


In [10]:
# I compare Google Docs phishing URLs with other test phishing URLs.

test_frame = partition_frames["Test"].copy()

test_frame["hostname"] = (
    test_frame["url"].map(get_hostname)
)

test_frame["has_query"] = test_frame["url"].map(
    lambda url: bool(urlsplit(str(url)).query)
)

test_frame["url_length"] = (
    test_frame["url"].str.len()
)

test_phishing = test_frame.loc[
    test_frame["label"] == 1
].copy()

test_phishing["group"] = np.where(
    test_phishing["hostname"] == "docs.google.com",
    "Google Docs",
    "Other phishing hostnames"
)

group_comparison = (
    test_phishing
    .groupby("group")
    .agg(
        records=("url", "size"),
        query_rate=("has_query", "mean"),
        mean_url_length=("url_length", "mean"),
        median_url_length=("url_length", "median")
    )
    .reset_index()
)

print("TEST PHISHING URL STRUCTURE COMPARISON")

display(group_comparison.round(4))

TEST PHISHING URL STRUCTURE COMPARISON


,group,records,query_rate,mean_url_length,median_url_length
0,Google Docs,3165,0.9788,173.9681,175.0
1,Other phishing hostnames,5056,0.0483,47.0847,38.0


### Hostname Group Size and Class Distribution

My investigation identified two unusually large phishing hostname groups.

The validation partition contains 3,220 phishing URLs from `cloudflare-ipfs.com`, while the test partition contains 3,165 phishing URLs from `docs.google.com`.

The Google Docs URLs are particularly different from other test phishing URLs because approximately 97.88% contain query parameters and their average length is substantially greater.

I now examine hostname group sizes across the complete dataset to determine whether these are isolated cases or part of a broader concentration pattern.

I preserve the original partitions throughout this investigation.

In [14]:
# I combine the existing partitions for retrospective analysis.
combined_urls = pd.concat(
    [
        frame.assign(partition=name)
        for name, frame in partition_frames.items()
    ],
    ignore_index=True
)

# I derive hostnames and query indicators without modifying saved data.
combined_urls["hostname"] = (
    combined_urls["url"].map(get_hostname)
)

combined_urls["has_query"] = (
    combined_urls["url"].map(
        lambda url: bool(urlsplit(str(url)).query)
    )
)

combined_urls["url_length"] = (
    combined_urls["url"].str.len()
)

# I summarize the distribution of records across hostnames.
hostname_group_summary = (
    combined_urls
    .groupby("hostname")
    .agg(
        records=("url", "size"),
        legitimate=("label", lambda x: (x == 0).sum()),
        phishing=("label", lambda x: (x == 1).sum()),
        query_rate=("has_query", "mean"),
        mean_url_length=("url_length", "mean"),
        partitions=("partition", "nunique")
    )
    .sort_values("records", ascending=False)
    .reset_index()
)

print("TOP 20 LARGEST HOSTNAME GROUPS")

display(
    hostname_group_summary.head(20).round(4)
)

# I measure how concentrated the complete dataset is.
total_records = len(combined_urls)

concentration_summary = pd.DataFrame([
    {
        "group_count": n,
        "records": int(
            hostname_group_summary.head(n)["records"].sum()
        ),
        "dataset_share": (
            hostname_group_summary.head(n)["records"].sum()
            / total_records
        )
    }
    for n in [1, 5, 10, 20, 50, 100]
])

print("\nDATASET CONCENTRATION")

display(concentration_summary.round(4))

# I verify that hostname separation was preserved.
assert hostname_group_summary["partitions"].max() == 1

print("\nHostname separation: PASSED")

TOP 20 LARGEST HOSTNAME GROUPS


,hostname,records,legitimate,phishing,query_rate,mean_url_length,partitions
0,cloudflare-ipfs.com,3220,0,3220,0.0649,101.5922,1
1,docs.google.com,3165,0,3165,0.9788,173.9681,1
2,ipfs.eth.aragon.network,2180,0,2180,0.0000,100.2229,1
3,cf-ipfs.com,1354,0,1354,0.0015,76.1743,1
4,new.express.adobe.com,949,0,949,0.0084,51.1496,1
5,tinyurl.com,932,282,650,0.1137,33.0343,1
6,ipfs.io,877,0,877,0.0068,71.8461,1
7,sites.google.com,658,0,658,0.0851,53.3784,1
8,bit.ly,457,288,169,0.0066,30.5777,1
9,forms.gle,313,296,17,0.0288,37.8722,1



DATASET CONCENTRATION


,group_count,records,dataset_share
0,1,3220,0.0248
1,5,10868,0.0837
2,10,14105,0.1087
3,20,17096,0.1317
4,50,25863,0.1993
5,100,39516,0.3045



Hostname separation: PASSED


### Review of the Hostname-Aware Splitting Strategy

My original splitting algorithm assigned entire hostname groups to training, validation, or test partitions while targeting a 70/15/15 class distribution.

The allocation successfully prevented identical hostnames from appearing in multiple partitions. However, it balanced legitimate and phishing record counts without explicitly considering structural differences between hostname groups.

My retrospective investigation revealed that the largest 100 hostnames account for approximately 30.45% of the dataset.

Several large phishing hostname groups also have substantially different URL characteristics.

This helps explain why the partitions achieved nearly identical phishing proportions but produced different distributions of query-containing phishing URLs.

I retain the original partitions and investigate how the largest hostname groups were allocated before deciding whether a revised evaluation protocol is necessary.

In [17]:
# I examine how the largest hostname groups were allocated.

largest_groups = (
    combined_urls
    .groupby(["hostname", "partition"])
    .agg(
        records=("url", "size"),
        legitimate=("label", lambda x: (x == 0).sum()),
        phishing=("label", lambda x: (x == 1).sum()),
        query_rate=("has_query", "mean"),
        mean_url_length=("url_length", "mean")
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

print("TOP 20 HOSTNAME GROUPS BY PARTITION")

display(
    largest_groups.head(20).round(4)
)

# I summarize the largest groups assigned to each partition.
top_groups_by_partition = (
    largest_groups
    .groupby("partition", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

print("\nTOP 5 HOSTNAME GROUPS IN EACH PARTITION")

display(
    top_groups_by_partition.round(4)
)

TOP 20 HOSTNAME GROUPS BY PARTITION


,hostname,partition,records,legitimate,phishing,query_rate,mean_url_length
10773,cloudflare-ipfs.com,Validation,3220,0,3220,0.0649,101.5922
12670,docs.google.com,Test,3165,0,3165,0.9788,173.9681
17212,ipfs.eth.aragon.network,Train,2180,0,2180,0.0000,100.2229
10347,cf-ipfs.com,Train,1354,0,1354,0.0015,76.1743
21367,new.express.adobe.com,Train,949,0,949,0.0084,51.1496
30831,tinyurl.com,Train,932,282,650,0.1137,33.0343
17214,ipfs.io,Train,877,0,877,0.0068,71.8461
28683,sites.google.com,Train,658,0,658,0.0851,53.3784
8646,bit.ly,Train,457,288,169,0.0066,30.5777
14557,forms.gle,Train,313,296,17,0.0288,37.8722



TOP 5 HOSTNAME GROUPS IN EACH PARTITION


,hostname,partition,records,legitimate,phishing,query_rate,mean_url_length
0,cloudflare-ipfs.com,Validation,3220,0,3220,0.0649,101.5922
1,docs.google.com,Test,3165,0,3165,0.9788,173.9681
2,ipfs.eth.aragon.network,Train,2180,0,2180,0.0000,100.2229
3,cf-ipfs.com,Train,1354,0,1354,0.0015,76.1743
4,new.express.adobe.com,Train,949,0,949,0.0084,51.1496
5,tinyurl.com,Train,932,282,650,0.1137,33.0343
6,ipfs.io,Train,877,0,877,0.0068,71.8461
7,issuu.com,Test,301,300,1,0.2027,59.3389
8,www.gstatic.com,Validation,300,300,0,0.0000,182.8667
9,www.miit.gov.cn,Test,299,299,0,0.1639,103.8361


### Post-Split Diagnostic Conclusions

My initial hostname-aware splitting strategy successfully separated hostnames across training, validation, and test partitions while maintaining nearly identical class proportions.

However, the final model evaluation revealed a substantial difference between validation and test performance. My Histogram Gradient Boosting classifier achieved 91.79% phishing F1-score on validation but only 67.50% on the held-out test partition.

I investigated this discrepancy and found that the test partition contained 3,165 URLs labeled phishing from `docs.google.com`, approximately 97.88% of which contained query parameters.

These records represented 38.5% of the test phishing class and contributed substantially to the difference in URL structures between the partitions.

The validation partition contained another large phishing group, `cloudflare-ipfs.com`, but its query usage was considerably lower.

I also found that the largest 100 hostnames accounted for approximately 30.45% of the complete dataset.

These findings showed me that preserving hostname separation and class proportions was not sufficient to produce structurally comparable partitions.

I therefore decided to retain the original partitions and their evaluation results as Version 1, while investigating a revised evaluation strategy that accounts for hostname-group concentration and structural diversity.

Any subsequent evaluation using these same records will be treated as exploratory because I have already examined the original test results.

My objective is not to obtain a more favorable score, but to establish a more reliable way of evaluating whether the model generalizes to unfamiliar URL structures.

## Version 2 — Revised Dataset Partitioning Strategy

My initial dataset partitions successfully preserved hostname separation and overall class balance, but they contained substantial differences in URL structure.

The most significant issue was the concentration of query-heavy phishing URLs within a single test hostname group.

I therefore develop a revised partitioning strategy with four objectives:

1. Preserve hostname separation to reduce memorization of host-specific patterns.
2. Maintain reasonable legitimate and phishing class proportions.
3. Account for large hostname groups and structural characteristics during partition construction.
4. Evaluate structurally unusual hostname groups separately rather than hiding their difficulty through redistribution.

I retain the original dataset partitions and evaluation results.

Because this revision follows an examination of the original test results, I treat Version 2 as exploratory development work. A genuinely independent evaluation dataset will be required for a fresh final performance estimate.

In [21]:
# I define the requirements for my revised dataset partitions.

V2_CONFIG = {
    "random_state": 42,

    "train_fraction": 0.70,
    "validation_fraction": 0.15,
    "evaluation_fraction": 0.15,

    "group_column": "hostname",
    "target_column": "label",

    "structural_features": [
        "has_query",
        "is_https",
        "url_length"
    ],

    "large_group_threshold": 500,

    "preserve_original_partitions": True,

    "evaluation_status": "retrospective_exploratory"
}

assert abs(
    V2_CONFIG["train_fraction"]
    + V2_CONFIG["validation_fraction"]
    + V2_CONFIG["evaluation_fraction"]
    - 1.0
) < 1e-10

print("VERSION 2 PARTITIONING CONFIGURATION")

for key, value in V2_CONFIG.items():
    print(f"{key}: {value}")

VERSION 2 PARTITIONING CONFIGURATION
random_state: 42
train_fraction: 0.7
validation_fraction: 0.15
evaluation_fraction: 0.15
group_column: hostname
target_column: label
structural_features: ['has_query', 'is_https', 'url_length']
large_group_threshold: 500
preserve_original_partitions: True
evaluation_status: retrospective_exploratory


### Large Hostname Group Profiling

I identify hostname groups containing at least 500 URLs to understand how they may influence dataset partitioning.

For each group, I examine its size, phishing proportion, HTTPS usage, query parameter frequency, and URL length.

I use these findings to establish appropriate constraints for my revised partitioning strategy.

The 500-record threshold is a diagnostic criterion rather than a reason to exclude or modify any records.

In [24]:
from urllib.parse import urlsplit

# I derive HTTPS usage from the original URL strings.
combined_urls["is_https"] = combined_urls["url"].map(
    lambda url: urlsplit(str(url)).scheme.lower() == "https"
)

# I summarize hostname groups using label and structural information.
hostname_profiles = (
    combined_urls
    .groupby("hostname")
    .agg(
        records=("url", "size"),
        legitimate=("label", lambda x: (x == 0).sum()),
        phishing=("label", lambda x: (x == 1).sum()),
        phishing_rate=("label", "mean"),
        https_rate=("is_https", "mean"),
        query_rate=("has_query", "mean"),
        mean_url_length=("url_length", "mean")
    )
    .reset_index()
)

# I identify groups meeting my large-group threshold.
large_hostname_groups = (
    hostname_profiles.loc[
        hostname_profiles["records"]
        >= V2_CONFIG["large_group_threshold"]
    ]
    .sort_values("records", ascending=False)
    .reset_index(drop=True)
)

print("LARGE HOSTNAME GROUPS")

print("Number of groups:", len(large_hostname_groups))
print("Total records:", int(large_hostname_groups["records"].sum()))

display(large_hostname_groups.round(4))

LARGE HOSTNAME GROUPS
Number of groups: 8
Total records: 13335


,hostname,records,legitimate,phishing,phishing_rate,https_rate,query_rate,mean_url_length
0,cloudflare-ipfs.com,3220,0,3220,1.0000,0.9960,0.0649,101.5922
1,docs.google.com,3165,0,3165,1.0000,0.9991,0.9788,173.9681
2,ipfs.eth.aragon.network,2180,0,2180,1.0000,1.0000,0.0000,100.2229
3,cf-ipfs.com,1354,0,1354,1.0000,0.9993,0.0015,76.1743
4,new.express.adobe.com,949,0,949,1.0000,0.9937,0.0084,51.1496
5,tinyurl.com,932,282,650,0.6974,0.8444,0.1137,33.0343
6,ipfs.io,877,0,877,1.0000,1.0000,0.0068,71.8461
7,sites.google.com,658,0,658,1.0000,0.9939,0.0851,53.3784


In [26]:
# I measure how much of each class belongs to large hostname groups.
large_hostnames = set(
    large_hostname_groups["hostname"]
)

combined_urls["is_large_group"] = (
    combined_urls["hostname"].isin(large_hostnames)
)

large_group_class_summary = (
    combined_urls
    .groupby(["label", "is_large_group"])
    .size()
    .unstack(fill_value=0)
)

large_group_class_summary.columns = [
    "Other hostname groups",
    "Large hostname groups"
]

large_group_class_summary["Total"] = (
    large_group_class_summary.sum(axis=1)
)

large_group_class_summary["Large group share"] = (
    large_group_class_summary["Large hostname groups"]
    / large_group_class_summary["Total"]
)

large_group_class_summary.index = [
    "Legitimate",
    "Phishing"
]

print("LARGE HOSTNAME GROUP CONTRIBUTION BY CLASS")

display(large_group_class_summary.round(4))

LARGE HOSTNAME GROUP CONTRIBUTION BY CLASS


,Other hostname groups,Large hostname groups,Total,Large group share
Legitimate,74688,282,74970,0.0038
Phishing,41751,13053,54804,0.2382


### Version 2 — Development and Challenge Dataset Separation

My large-hostname analysis revealed that eight hostnames contained at least 500 URLs.

These groups accounted for 23.82% of all phishing records but only 0.38% of legitimate records.

This concentration explained why allocating large hostname groups directly into ordinary validation and test partitions could produce substantially different structural distributions.

For Version 2, I separate the dataset into two components.

The first contains hostname groups with fewer than 500 records. I use these records to construct revised hostname-disjoint development partitions.

The second contains all eight large hostname groups. I retain these records as a separate challenge dataset for evaluating generalization to unfamiliar, high-volume hosting platforms.

I apply the same group-size criterion to every hostname rather than selecting individual groups based on model performance.

Because this investigation follows my examination of the original test results, I treat Version 2 as a retrospective exploratory experiment.

I preserve the original partitions and report their results separately.

In [29]:
# I identify large hostnames using the predefined threshold.
large_hostnames = set(
    large_hostname_groups["hostname"]
)

# I create independent working copies.
development_urls = combined_urls.loc[
    ~combined_urls["hostname"].isin(large_hostnames)
].copy()

challenge_urls = combined_urls.loc[
    combined_urls["hostname"].isin(large_hostnames)
].copy()

# I reset indices for subsequent processing.
development_urls = development_urls.reset_index(drop=True)
challenge_urls = challenge_urls.reset_index(drop=True)

print("VERSION 2 DATASET SEPARATION")

print("\nDevelopment dataset")
print("Records:", len(development_urls))
print("Unique hostnames:", development_urls["hostname"].nunique())

print("\nClass distribution:")
print(
    development_urls["label"]
    .value_counts()
    .sort_index()
)

print("\nChallenge dataset")
print("Records:", len(challenge_urls))
print("Unique hostnames:", challenge_urls["hostname"].nunique())

print("\nClass distribution:")
print(
    challenge_urls["label"]
    .value_counts()
    .sort_index()
)

# I ensure the two components contain no overlapping hostnames.
development_hostnames = set(development_urls["hostname"])
challenge_hostnames = set(challenge_urls["hostname"])

assert development_hostnames.isdisjoint(challenge_hostnames)

# I verify that every original record is accounted for.
assert (
    len(development_urls) + len(challenge_urls)
    == len(combined_urls)
)

assert len(challenge_hostnames) == len(large_hostname_groups)

print("\nHostname separation: PASSED")
print("Record preservation: PASSED")

VERSION 2 DATASET SEPARATION

Development dataset
Records: 116439
Unique hostnames: 37347

Class distribution:
label
0    74688
1    41751
Name: count, dtype: int64

Challenge dataset
Records: 13335
Unique hostnames: 8

Class distribution:
label
0      282
1    13053
Name: count, dtype: int64

Hostname separation: PASSED
Record preservation: PASSED


### Constructing the Revised Development Partitions

I construct hostname-aware training, validation, and exploratory evaluation partitions from the development dataset.

I use GroupShuffleSplit to keep URLs belonging to the same hostname together.

Because hostname groups contain different numbers of records, splitting groups randomly does not guarantee balanced record counts or URL characteristics.

I therefore generate multiple candidate partitions and select the configuration that best matches predefined targets for record proportions, class balance, query usage, and HTTPS usage.

I do not use model predictions or classification performance to select the partitions.

This procedure is retrospective and exploratory because I have already examined the original test results.

In [32]:
from sklearn.model_selection import GroupShuffleSplit

# I define the target proportions for my development partitions.
TARGET_FRACTIONS = {
    "Train": 0.70,
    "Validation": 0.15,
    "Evaluation": 0.15
}

# I use multiple deterministic candidates to examine split balance.
N_CANDIDATES = 30

def create_group_split(data, seed):
    """
    Create hostname-disjoint 70/15/15 development partitions.
    """

    groups = data["hostname"]

    # I first separate approximately 70% of hostname groups.
    first_split = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=seed
    )

    train_idx, remaining_idx = next(
        first_split.split(
            data,
            y=data["label"],
            groups=groups
        )
    )

    train = data.iloc[train_idx].copy()
    remaining = data.iloc[remaining_idx].copy()

    # I divide the remaining hostname groups equally.
    second_split = GroupShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=seed + 1000
    )

    val_idx, eval_idx = next(
        second_split.split(
            remaining,
            y=remaining["label"],
            groups=remaining["hostname"]
        )
    )

    validation = remaining.iloc[val_idx].copy()
    evaluation = remaining.iloc[eval_idx].copy()

    return {
        "Train": train,
        "Validation": validation,
        "Evaluation": evaluation
    }


def calculate_split_balance(partitions):
    """
    Measure deviations from predefined partition targets.
    """

    reference = pd.concat(
        partitions.values(),
        ignore_index=True
    )

    overall_phishing_rate = reference["label"].mean()

    balance_score = 0.0
    diagnostics = []

    for name, frame in partitions.items():

        target_fraction = TARGET_FRACTIONS[name]

        actual_fraction = len(frame) / len(reference)

        phishing_rate = frame["label"].mean()

        # I penalize deviations from target record proportions.
        balance_score += (
            (actual_fraction - target_fraction) ** 2
        ) * 10

        # I penalize deviations from the overall phishing rate.
        balance_score += (
            phishing_rate - overall_phishing_rate
        ) ** 2

        # I compare structural distributions within each class.
        for label_value in [0, 1]:

            reference_class = reference.loc[
                reference["label"] == label_value
            ]

            partition_class = frame.loc[
                frame["label"] == label_value
            ]

            if partition_class.empty:
                balance_score += 10
                continue

            for feature in ["has_query", "is_https"]:

                reference_rate = reference_class[
                    feature
                ].mean()

                partition_rate = partition_class[
                    feature
                ].mean()

                balance_score += (
                    partition_rate - reference_rate
                ) ** 2

        diagnostics.append({
            "partition": name,
            "records": len(frame),
            "record_fraction": actual_fraction,
            "phishing_rate": phishing_rate
        })

    return balance_score, diagnostics


# I evaluate reproducible candidate splits.
candidate_results = []

best_score = float("inf")
best_seed = None
best_partitions = None

for candidate in range(N_CANDIDATES):

    seed = V2_CONFIG["random_state"] + candidate

    partitions = create_group_split(
        development_urls,
        seed
    )

    score, diagnostics = calculate_split_balance(
        partitions
    )

    candidate_results.append({
        "seed": seed,
        "balance_score": score
    })

    if score < best_score:

        best_score = score
        best_seed = seed
        best_partitions = partitions


candidate_comparison = (
    pd.DataFrame(candidate_results)
    .sort_values("balance_score")
    .reset_index(drop=True)
)

print("VERSION 2 SPLIT SELECTION")

print("Candidates evaluated:", N_CANDIDATES)
print("Selected random seed:", best_seed)
print(f"Balance score: {best_score:.6f}")

print("\nTOP 10 CANDIDATES")

display(candidate_comparison.head(10).round(6))

VERSION 2 SPLIT SELECTION
Candidates evaluated: 30
Selected random seed: 64
Balance score: 0.002063

TOP 10 CANDIDATES


,seed,balance_score
0,64,0.002063
1,65,0.003199
2,43,0.003561
3,51,0.003871
4,44,0.004265
5,48,0.004309
6,53,0.006120
7,54,0.006444
8,70,0.006451
9,58,0.007410


In [34]:
# I inspect the selected development partitions.

v2_partition_summary = []

for name, frame in best_partitions.items():

    v2_partition_summary.append({
        "partition": name,
        "records": len(frame),
        "dataset_fraction": (
            len(frame) / len(development_urls)
        ),
        "unique_hostnames": frame["hostname"].nunique(),
        "legitimate": int((frame["label"] == 0).sum()),
        "phishing": int((frame["label"] == 1).sum()),
        "phishing_rate": frame["label"].mean()
    })

v2_partition_summary = pd.DataFrame(
    v2_partition_summary
)

print("VERSION 2 DEVELOPMENT PARTITIONS")

display(v2_partition_summary.round(4))


# I verify that all development records are preserved.
assert sum(
    len(frame) for frame in best_partitions.values()
) == len(development_urls)


# I verify that hostnames are disjoint.
train_hosts = set(best_partitions["Train"]["hostname"])
val_hosts = set(best_partitions["Validation"]["hostname"])
eval_hosts = set(best_partitions["Evaluation"]["hostname"])

assert train_hosts.isdisjoint(val_hosts)
assert train_hosts.isdisjoint(eval_hosts)
assert val_hosts.isdisjoint(eval_hosts)


# I verify that challenge hostnames remain excluded.
assert train_hosts.isdisjoint(challenge_hostnames)
assert val_hosts.isdisjoint(challenge_hostnames)
assert eval_hosts.isdisjoint(challenge_hostnames)

print("\nRecord preservation: PASSED")
print("Development hostname separation: PASSED")
print("Challenge hostname separation: PASSED")

VERSION 2 DEVELOPMENT PARTITIONS


,partition,records,dataset_fraction,unique_hostnames,legitimate,phishing,phishing_rate
0,Train,82357,0.7073,26142,52807,29550,0.3588
1,Validation,17702,0.1520,5602,11495,6207,0.3506
2,Evaluation,16380,0.1407,5603,10386,5994,0.3659



Record preservation: PASSED
Development hostname separation: PASSED
Challenge hostname separation: PASSED


### Version 2 — Structural Distribution Validation

I generated 30 reproducible hostname-aware partition candidates and selected seed 64 based on predefined balance criteria.

The selected partitions contain approximately 70.73% training, 15.20% validation, and 14.07% exploratory evaluation records.

All partitions preserve hostname separation and maintain similar phishing proportions.

Before accepting the partitions, I compare URL characteristics within each class, including query usage, HTTPS usage, URL length, hostname length, and path length.

My objective is to identify substantial distribution differences that could affect model evaluation. I do not expect the partitions to have identical characteristics, but I want to avoid the extreme imbalances observed in my original dataset split.

In [37]:
# I compare URL structures across the selected development partitions.

v2_structure_results = []

for partition_name, frame in best_partitions.items():

    for label_value, class_name in [
        (0, "Legitimate"),
        (1, "Phishing")
    ]:

        subset = frame.loc[
            frame["label"] == label_value
        ]

        # I derive URL components consistently.
        parsed_urls = subset["url"].map(
            lambda url: urlsplit(str(url))
        )

        url_lengths = subset["url"].str.len()

        hostname_lengths = parsed_urls.map(
            lambda parsed: len(parsed.hostname or "")
        )

        path_lengths = parsed_urls.map(
            lambda parsed: len(parsed.path)
        )

        query_lengths = parsed_urls.map(
            lambda parsed: len(parsed.query)
        )

        v2_structure_results.append({
            "partition": partition_name,
            "class": class_name,
            "records": len(subset),
            "https_rate": subset["is_https"].mean(),
            "query_rate": subset["has_query"].mean(),
            "nonempty_path_rate": (
                path_lengths > 0
            ).mean(),
            "mean_url_length": url_lengths.mean(),
            "mean_hostname_length": hostname_lengths.mean(),
            "mean_path_length": path_lengths.mean(),
            "mean_query_length": query_lengths.mean()
        })


v2_structure_comparison = pd.DataFrame(
    v2_structure_results
)

print("VERSION 2 STRUCTURAL DISTRIBUTIONS")

display(
    v2_structure_comparison.round(4)
)

VERSION 2 STRUCTURAL DISTRIBUTIONS


,partition,class,records,https_rate,query_rate,nonempty_path_rate,mean_url_length,mean_hostname_length,mean_path_length,mean_query_length
0,Train,Legitimate,52807,0.8755,0.2749,0.9871,86.9439,17.0799,33.1089,28.3757
1,Train,Phishing,29550,0.8638,0.0651,0.8663,55.0014,28.9645,9.3717,6.5448
2,Validation,Legitimate,11495,0.8698,0.2824,0.9907,84.3120,17.1688,33.0130,25.8647
3,Validation,Phishing,6207,0.8735,0.0806,0.8684,52.2885,28.7359,7.2502,8.0867
4,Evaluation,Legitimate,10386,0.8707,0.2764,0.9919,92.4119,17.0372,34.1718,33.0014
5,Evaluation,Phishing,5994,0.8700,0.0854,0.8652,55.0864,29.4887,8.4768,9.0132


In [39]:
# I calculate the range of each structural characteristic
# across development partitions, separately for each class.

structural_metrics = [
    "https_rate",
    "query_rate",
    "nonempty_path_rate",
    "mean_url_length",
    "mean_hostname_length",
    "mean_path_length",
    "mean_query_length"
]

v2_distribution_ranges = (
    v2_structure_comparison
    .groupby("class")[structural_metrics]
    .agg(lambda values: values.max() - values.min())
    .reset_index()
)

print("MAXIMUM BETWEEN-PARTITION DIFFERENCES")

display(
    v2_distribution_ranges.round(4)
)

MAXIMUM BETWEEN-PARTITION DIFFERENCES


,class,https_rate,query_rate,nonempty_path_rate,mean_url_length,mean_hostname_length,mean_path_length,mean_query_length
0,Legitimate,0.0057,0.0075,0.0048,8.0999,0.1316,1.1588,7.1367
1,Phishing,0.0097,0.0203,0.0032,2.7979,0.7527,2.1215,2.4683


### Saving the Revised Dataset Partitions

I save the revised hostname-disjoint development partitions and the large-hostname challenge dataset as a separate dataset version.

I use a new filename prefix to preserve my original URL partitions and maintain reproducibility.

The development partitions contain hostname groups with fewer than 500 records, while the challenge dataset contains the eight larger hostname groups.

I retain the selected random seed, partition sizes, class distributions, structural validation results, and methodology limitations in a metadata file.

I classify all Version 2 evaluations as retrospective and exploratory because the revised strategy was developed after inspecting the original test results.

In [42]:
# I save the revised partitions without overwriting earlier datasets.

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

v3_datasets = {
    "url_v3_train.csv": best_partitions["Train"],
    "url_v3_val.csv": best_partitions["Validation"],
    "url_v3_eval.csv": best_partitions["Evaluation"],
    "url_v3_challenge.csv": challenge_urls
}

# I retain only the columns needed for modeling and auditing.
EXPORT_COLUMNS = [
    "url",
    "label",
    "source"
]

for filename, frame in v3_datasets.items():

    output_path = SPLIT_DIR / filename

    assert not output_path.exists(), (
        f"File already exists: {output_path}"
    )

    frame[EXPORT_COLUMNS].to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved: {filename} "
        f"({len(frame):,} records)"
    )

print("\nAll Version 3 dataset files saved.")

Saved: url_v3_train.csv (82,357 records)
Saved: url_v3_val.csv (17,702 records)
Saved: url_v3_eval.csv (16,380 records)
Saved: url_v3_challenge.csv (13,335 records)

All Version 3 dataset files saved.


In [48]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

metadata_path = PROCESSED_DIR / "url_v3_metadata.json"

In [50]:
import json

# I document how the revised dataset was constructed.

v3_metadata = {
    "dataset_version": "v3",
    "construction_notebook": "05_url_dataset_construction.ipynb",
    "source_dataset": "Wangchuk_2026_CommonCrawl_Phishtank",

    "evaluation_status": "retrospective_exploratory",

    "original_dataset_records": int(len(combined_urls)),

    "development_records": int(len(development_urls)),
    "challenge_records": int(len(challenge_urls)),

    "large_hostname_threshold": int(
        V2_CONFIG["large_group_threshold"]
    ),

    "large_hostname_count": int(
        len(large_hostname_groups)
    ),

    "selected_random_seed": int(best_seed),
    "candidate_splits_evaluated": int(N_CANDIDATES),
    "selected_balance_score": float(best_score),

    "target_fractions": TARGET_FRACTIONS,

    "partition_summary": (
        v2_partition_summary.to_dict(orient="records")
    ),

    "structural_distribution_summary": (
        v2_structure_comparison.to_dict(orient="records")
    ),

    "limitations": [
        "Partitions were revised after inspecting original test results.",
        "Development partitions exclude hostname groups containing at least 500 records.",
        "The challenge dataset has a heavily imbalanced class distribution.",
        "Hostname separation does not guarantee independence between related domains or campaigns.",
        "A fresh independent evaluation dataset is needed for an unbiased final performance estimate."
    ]
}

metadata_path = (
    PROCESSED_DIR / "url_v3_metadata.json"
)

assert not metadata_path.exists(), (
    f"Metadata already exists: {metadata_path}"
)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        v3_metadata,
        file,
        indent=2
    )

print("Metadata saved:", metadata_path)

Metadata saved: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\data\processed\url_v3_metadata.json


In [52]:
# I verify that saved datasets match the in-memory partitions.

for filename, original_frame in v3_datasets.items():

    saved_path = SPLIT_DIR / filename

    saved_frame = pd.read_csv(saved_path)

    assert len(saved_frame) == len(original_frame)

    assert saved_frame.columns.tolist() == EXPORT_COLUMNS

    assert (
        saved_frame["label"].value_counts().sort_index()
        .equals(
            original_frame["label"]
            .value_counts()
            .sort_index()
        )
    )

    print(f"{filename}: VERIFIED")

assert metadata_path.is_file()

print("\nVERSION 3 DATASET VERIFICATION: PASSED")

url_v3_train.csv: VERIFIED
url_v3_val.csv: VERIFIED
url_v3_eval.csv: VERIFIED
url_v3_challenge.csv: VERIFIED

VERSION 3 DATASET VERIFICATION: PASSED
